# 1. Data Preparation

**Pipeline position:** first prep notebook. Reads the cleaned panel from the
main capstone repo (`Master.csv`, `clusters1995.csv`) and produces the panel
that every chart downstream consumes.

## What this notebook does

1. Loads `Master.csv` and applies the forest-adjusted sample selection
   (subtract forestry rents from total NR rents, keep countries with
   adjusted rents at or above 1% of GDP, force-include Gulf states).
2. Restricts to 1995–2019 and merges k=5 cluster labels.
3. Adds the derived columns the charts need (log GDP per capita, production
   per capita, scaled bubble size for the animated scatter, lagged ECI,
   log transforms, mean-centred interaction terms).
4. Saves the resulting panel as `panel.parquet` and the 1995 cluster
   assignments as `clusters_1995.csv`.

## Outputs

- `artifacts/panel.parquet` — country–year panel for the 38-country sample,
  with all derived columns. Used by every other notebook and every chart.
- `artifacts/clusters_1995.csv` — `Country Code, Country, ClusterLabels`
  for the cluster world map (chart 5).
- `artifacts/sample.csv` — the list of 38 country codes, written so the
  robustness notebook can rebuild larger samples without re-doing this work.

Run from `capstone_viz/prep/`. Expects `_config.py` in the same folder.


In [1]:
# ── Imports and config ──────────────────────────────────────────────────────
import os, sys
sys.path.insert(0, os.path.dirname(os.path.abspath('')) + '/prep')
sys.path.insert(0, '.')

import numpy as np
import pandas as pd

from _config import (
    INTERMEDIARY, ARTIFACTS, GULF_STATES, NR_THRESHOLD,
    YEAR_MIN, YEAR_MAX, LOG_COLS, ECI_COL,
    artifact_path, load_master, load_clusters_1995,
)

print(f'Reading from: {INTERMEDIARY}')
print(f'Writing to:   {ARTIFACTS}')


Reading from: /Users/leoss/Desktop/GitHub/Portfolio/projects/capstone/New code/intermediary
Writing to:   /Users/leoss/Desktop/GitHub/Portfolio/projects/capstone/artifacts


## Sample selection

The sample is defined on the 1995 snapshot. For each country we compute
adjusted natural-resource rents (total NR rents minus forestry rents,
clipped at zero) and keep those at or above 1% of GDP. Gulf states are
guaranteed inclusion because their adjusted rents at 1995 exclude oil
sectors that come online later in the panel.


In [2]:
master = load_master()

base_1995 = master[master['Year'] == 1995].copy()
base_1995['NR_adj'] = (
    base_1995['Total natural resources rents (% of GDP)'].fillna(0)
    - base_1995['Forestry rents (% of GDP)'].fillna(0)
).clip(lower=0)

include_lst = base_1995.loc[
    base_1995['NR_adj'] >= NR_THRESHOLD, 'Country Code'
].unique().tolist()

for g in GULF_STATES:
    if g not in include_lst:
        include_lst.append(g)

print(f'Forest-adjusted sample (≥{NR_THRESHOLD:.0f}% + Gulf): '
      f'{len(include_lst)} countries')


Forest-adjusted sample (≥1% + Gulf): 49 countries


## Build the panel

Restrict `Master.csv` to the sample, attach cluster labels, add the columns
charts will need, and save.


In [3]:
clusters_1995 = load_clusters_1995()
cl_map = clusters_1995[['Country Code', 'ClusterLabels']].drop_duplicates('Country Code')

panel = master[
    master['Year'].between(YEAR_MIN, YEAR_MAX)
    & master['Country Code'].isin(include_lst)
].copy()

panel = panel.sort_values(['Country Code', 'Year']).reset_index(drop=True)
panel = panel.merge(cl_map, on='Country Code', how='left')

# Derived columns used by the descriptive and animation charts ----------------
panel['Log_GDP_pc'] = np.log(
    panel['GDP per capita (constant prices, PPP)'].replace(0, np.nan)
)
panel['Total_Production_Value_Per_Capita'] = (
    panel['Total_Production_Value'] / panel['Population'].replace(0, np.nan)
)
panel['Prod_pc'] = panel['Total_Production_Value_Per_Capita']
panel['Bubble']  = np.sqrt(panel['Prod_pc'].fillna(0))

bmin, bmax = panel['Bubble'].min(), panel['Bubble'].max()
panel['Bubble_Scaled'] = 8 + (panel['Bubble'] - bmin) / (bmax - bmin + 1e-9) * 42

# Lagged ECI ------------------------------------------------------------------
panel['L1_ECI'] = panel.groupby('Country Code')[ECI_COL].shift(1)

# Log-transform features used by the ML and regression notebooks --------------
for c in LOG_COLS:
    if c in panel.columns:
        panel[c] = np.log1p(panel[c]).replace([np.inf, -np.inf], np.nan)

# Aliases used by the regression notebook (already log-transformed above) -----
panel['log_HCI']             = panel['Human capital index']
panel['log_GFCF']            = panel['Gross fixed capital formation, all, Constant prices, Percent of GDP']
panel['log_Production_Value']= panel['Total_Production_Value_Per_Capita']

# Mean-centred interaction terms (regression notebook + ML notebook) ----------
hci_m  = panel['log_HCI'].mean()
gfcf_m = panel['log_GFCF'].mean()
prod_m = panel['log_Production_Value'].mean()
frt_m  = panel['Forestry rents (% of GDP)'].mean()

panel['HCI_x_ProductionValue']  = (panel['log_HCI']  - hci_m)  * (panel['log_Production_Value'] - prod_m)
panel['GFCF_x_ProductionValue'] = (panel['log_GFCF'] - gfcf_m) * (panel['log_Production_Value'] - prod_m)
panel['log_HCI_x_log_Production']  = panel['HCI_x_ProductionValue']
panel['log_GFCF_x_log_Production'] = panel['GFCF_x_ProductionValue']
panel['log_HCI_x_forestry_rents']  = (panel['log_HCI']  - hci_m)  * (panel['Forestry rents (% of GDP)'] - frt_m)
panel['log_GFCF_x_forestry_rents'] = (panel['log_GFCF'] - gfcf_m) * (panel['Forestry rents (% of GDP)'] - frt_m)

print(f'Panel: {panel["Country Code"].nunique()} countries, '
      f'{len(panel):,} obs, {YEAR_MIN}-{YEAR_MAX}')


Panel: 49 countries, 1,225 obs, 1995-2019


## Save artifacts

Three files. The parquet is the heavy one (panel) and is read by every
other prep notebook. The two CSVs are tiny and used directly by the viz
notebooks.


In [4]:
panel.to_parquet(artifact_path('panel.parquet'), index=False)
print(f'  Saved panel.parquet ({len(panel):,} rows, {panel.shape[1]} cols)')

clusters_1995[['Country Code', 'Country', 'ClusterLabels']] \
    .drop_duplicates('Country Code') \
    .to_csv(artifact_path('clusters_1995.csv'), index=False)
print(f'  Saved clusters_1995.csv')

pd.Series(include_lst, name='Country Code') \
    .to_csv(artifact_path('sample.csv'), index=False)
print(f'  Saved sample.csv ({len(include_lst)} countries)')


  Saved panel.parquet (1,225 rows, 65 cols)
  Saved clusters_1995.csv
  Saved sample.csv (49 countries)


## Summary

| Step | Result |
|---|---|
| Master.csv loaded | full panel, all countries, 1995–2019 |
| Forest-adjusted threshold (≥1% + Gulf) | 38 countries kept |
| Derived columns added | log GDP per capita, production per capita, lagged ECI, log transforms, 4 interaction terms |
| Saved | panel.parquet, clusters_1995.csv, sample.csv |

Run notebook 2 (`2_clusters.ipynb`) next.
